# derived_8.4-hybrid-lstm-1.2 — Hybrid LSTM CTX + Head Hidden + XGBoost with SHAP

This experiment evaluates the **Hybrid LSTM + XGBoost** architecture on the Washington-only dataset (7 stations, 2023–2025 test set), comparing two CTX representations extracted from the same trained BiLSTM+Attn model:
- **160-dim `ctx`**: Full attention-pooled hidden state
- **80-dim `head_hidden`**: Intermediate after head `Linear(160→80)→ReLU`

### Hypothesis
The 160-dim CTX may be too large, diluting feature importance across too many dimensions. The 80-dim head bottleneck may provide a more compressed, higher-signal representation.

### Architecture & Pipeline
1. **Phase 1 (BiLSTM Training)**: Train BiLSTM+Attn v9 on sequence dataset until early-stopping convergence.
2. **Phase 2 (Frozen Extraction)**: Extract both 160-dim `ctx` and 80-dim `head_hidden` vectors.
3. **Phase 3 (XGBoost Hybrid Fusion)**: Evaluate 6 models — 2 pure tabular baselines + 4 hybrid models (2× CTX-dim).
4. **Phase 4 (Accelerated SHAP Analysis)**: Compute SHAP values using XGBoost native `pred_contribs=True`.


In [1]:
import sys
from pathlib import Path
import json
import numpy as np
import pandas as pd
import yaml

EXP_DIR = Path("experiment/derived_8.4-hybrid-lstm-1.2").resolve() if Path("experiment/derived_8.4-hybrid-lstm-1.2").exists() else Path(".").resolve()
PROJECT_ROOT = EXP_DIR.parents[2]
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(EXP_DIR))

print(f"[Setup] Project Root: {PROJECT_ROOT}")
print(f"[Setup] Experiment Directory: {EXP_DIR}")


[Setup] Project Root: /scratch/user/u.rp352032/MDR-Project
[Setup] Experiment Directory: /scratch/user/u.rp352032/MDR-Project/notebooks/experiment/derived_8.4-hybrid-lstm-1.2


## Phase 1 & Phase 2: BiLSTM Training & Frozen CTX + Head Hidden Extraction

Train the `BiLSTMAttn` model on `derived_8.4` until early-stopping convergence, then freeze and extract two representations:
- **`ctx`** (160-dim): Full attention-pooled hidden state from the BiLSTM
- **`head_hidden`** (80-dim): Intermediate after head `Linear(160→80)→ReLU` — a more compressed bottleneck representation


In [2]:
import sys, json, numpy as np, pandas as pd
from pathlib import Path
from lstm.train import train_lstm_and_extract_ctx

EXP_DIR = Path("experiment/derived_8.4-hybrid-lstm-1.2").resolve() if Path("experiment/derived_8.4-hybrid-lstm-1.2").exists() else Path(".").resolve()
PROJECT_ROOT = EXP_DIR.parents[2]
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(EXP_DIR))

data_dir = PROJECT_ROOT / "data/splits/derived_8.4"
artifacts_dir = EXP_DIR / "artifacts"

need_extract = not (
    (artifacts_dir / "ctx_test.npy").exists()
    and (artifacts_dir / "head_hidden_test.npy").exists()
    and (artifacts_dir / "lstm_metrics.json").exists()
)

if need_extract:
    print("[LSTM] Training BiLSTM+Attn and extracting CTX + Head Hidden representations...")
    train_lstm_and_extract_ctx(data_dir, artifacts_dir)
else:
    print("[LSTM] Loading pre-extracted frozen CTX + Head Hidden representations from artifacts...")

ctx_tr = np.load(artifacts_dir / "ctx_train.npy")
ctx_va = np.load(artifacts_dir / "ctx_val.npy")
ctx_te = np.load(artifacts_dir / "ctx_test.npy")

hh_tr = np.load(artifacts_dir / "head_hidden_train.npy")
hh_va = np.load(artifacts_dir / "head_hidden_val.npy")
hh_te = np.load(artifacts_dir / "head_hidden_test.npy")

with open(artifacts_dir / "lstm_metrics.json") as f:
    lstm_metrics = json.load(f)

print(f"\n[LSTM Phase Complete]")
print(f"  CTX (160-dim):      Train {ctx_tr.shape}, Val {ctx_va.shape}, Test {ctx_te.shape}")
print(f"  Head Hidden (80-dim): Train {hh_tr.shape}, Val {hh_va.shape}, Test {hh_te.shape}")
print(f"[LSTM Test Performance] R2 = {lstm_metrics['test']['r2']:.4f}, RMSE = {lstm_metrics['test']['rmse']:.5f}")


[LSTM] Loading pre-extracted frozen CTX + Head Hidden representations from artifacts...



[LSTM Phase Complete]
  CTX (160-dim):      Train (9803, 160), Val (4805, 160), Test (6620, 160)
  Head Hidden (80-dim): Train (9803, 80), Val (4805, 80), Test (6620, 80)
[LSTM Test Performance] R2 = 0.6186, RMSE = 0.06291


## Phase 3: XGBoost Hybrid Modeling & Evaluation

We evaluate **six models** on the `derived_8.4` test set (6,620 samples):

| # | Model | Features |
|---|-------|----------|
| 1 | Global Single (54 Backbone) | 54 tabular |
| 2 | Clustering_V0_Full_k2 (c0=0, c1=10) | 54/64 tabular |
| 3 | Global Single (54 Backbone + 160 CTX) | 214 |
| 4 | Clustering_V0_Full_k2 + 160 CTX | 214/224 |
| 5 | **Global Single (54 Backbone + 80 CTX-head)** | **134** |
| 6 | **Clustering_V0_Full_k2 + 80 CTX-head** | **134/144** |

Models 5–6 use the 80-dim intermediate head representation to test whether d=160 is too large.


In [3]:
import sys, json, yaml, numpy as np, pandas as pd
from pathlib import Path

EXP_DIR = Path("experiment/derived_8.4-hybrid-lstm-1.2").resolve() if Path("experiment/derived_8.4-hybrid-lstm-1.2").exists() else Path(".").resolve()
PROJECT_ROOT = EXP_DIR.parents[2]
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(EXP_DIR))

from eval_hybrid.data import load_hybrid_experiment_data
from eval_hybrid.evaluator import HybridStrategyEvaluator
from run_eval import compute_c1_gain_additions

artifacts_dir = EXP_DIR / "artifacts"
models_dir = EXP_DIR / "models"

with open(EXP_DIR / "config.yaml") as f:
    config = yaml.safe_load(f)

data = load_hybrid_experiment_data(PROJECT_ROOT, EXP_DIR, config)
c1_additions = compute_c1_gain_additions(config)

print(f"[Data] {len(data.shared_backbone_54)} backbone + {len(data.ctx_feature_cols)} CTX (160) + {len(data.ctx_80_feature_cols)} CTX-head (80)")
print(f"[Hybrid] backbone_214 = {len(data.hybrid_backbone_214)} | backbone_134 = {len(data.hybrid_backbone_134)}")

summary_records = []

eval_global = HybridStrategyEvaluator(data, config, "Global_Single", models_dir=models_dir)
eval_v0 = HybridStrategyEvaluator(data, config, "Clustering_V0_Full_k2", models_dir=models_dir)

# 1. Global Single Baseline (54 Backbone)
res_g_base = eval_global.fit_and_evaluate("Global Single Model (54 Backbone)", "Global_Single_54_Backbone", data.shared_backbone_54)
summary_records.append(res_g_base.as_record())

# 2. Clustering_V0_Full_k2 Baseline
res_v0_base = eval_v0.fit_and_evaluate("Clustering_V0_Full_k2 (Winner c0=0, c1=10)", "Clustering_V0_Full_k2_c0_0_c1_10", data.shared_backbone_54, {"0": [], "1": c1_additions})
summary_records.append(res_v0_base.as_record())

# 3. Global Single Hybrid (160 CTX)
res_g_hybrid = eval_global.fit_and_evaluate("Global Single Model (54 Backbone + 160 CTX)", "Global_Single_54_Backbone_160_CTX", data.hybrid_backbone_214)
summary_records.append(res_g_hybrid.as_record())

# 4. Clustering_V0_Full_k2 Hybrid (160 CTX)
res_v0_hybrid = eval_v0.fit_and_evaluate("Clustering_V0_Full_k2 (Winner c0=0, c1=10 + 160 CTX)", "Clustering_V0_Full_k2_c0_0_c1_10_160_CTX", data.hybrid_backbone_214, {"0": [], "1": c1_additions})
summary_records.append(res_v0_hybrid.as_record())

# 5. Global Single Hybrid (80 CTX-head)
res_g_hybrid_80 = eval_global.fit_and_evaluate("Global Single Model (54 Backbone + 80 CTX-head)", "Global_Single_54_Backbone_80_CTXhead", data.hybrid_backbone_134)
summary_records.append(res_g_hybrid_80.as_record())

# 6. Clustering_V0_Full_k2 Hybrid (80 CTX-head)
res_v0_hybrid_80 = eval_v0.fit_and_evaluate("Clustering_V0_Full_k2 (Winner c0=0, c1=10 + 80 CTX-head)", "Clustering_V0_Full_k2_c0_0_c1_10_80_CTXhead", data.hybrid_backbone_134, {"0": [], "1": c1_additions})
summary_records.append(res_v0_hybrid_80.as_record())

df_summary = pd.DataFrame(summary_records).sort_values("pooled_r2", ascending=False)
results_map = {
    "Global Single Model (54 Backbone)": res_g_base,
    "Clustering_V0_Full_k2 (Winner c0=0, c1=10)": res_v0_base,
    "Global Single Model (54 Backbone + 160 CTX)": res_g_hybrid,
    "Clustering_V0_Full_k2 (Winner c0=0, c1=10 + 160 CTX)": res_v0_hybrid,
    "Global Single Model (54 Backbone + 80 CTX-head)": res_g_hybrid_80,
    "Clustering_V0_Full_k2 (Winner c0=0, c1=10 + 80 CTX-head)": res_v0_hybrid_80,
}


[Data] 54 backbone + 160 CTX (160) + 80 CTX-head (80)
[Hybrid] backbone_214 = 214 | backbone_134 = 134


/scratch/user/u.rp352032/MDR-Project/notebooks/.venv/lib64/python3.12/site-packages/xgboost/core.py:751: UserWarning: [12:36:37] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


## Results & Diagnostic Summary

Leaderboard comparison across all 6 models: Pooled $R^2$, RMSE, ubRMSE, Bias, MAE, and Pearson correlation.


In [4]:
# Display styled summary dataframe
df_summary[["model_name", "pooled_r2", "pooled_rmse", "pooled_ubrmse", "pooled_bias", "pooled_mae", "pooled_pearson"]]


,model_name,pooled_r2,pooled_rmse,pooled_ubrmse,pooled_bias,pooled_mae,pooled_pearson
1,"Clustering_V0_Full_k2 (Winner c0=0, c1=10)",0.814960,0.043820,0.043337,0.006486,0.033719,0.905594
0,Global Single Model (54 Backbone),0.779230,0.047864,0.046687,0.010548,0.037059,0.889432
3,"Clustering_V0_Full_k2 (Winner c0=0, c1=10 + 16...",0.772048,0.048636,0.047864,0.008633,0.036382,0.886389
2,Global Single Model (54 Backbone + 160 CTX),0.760117,0.049893,0.048944,0.009683,0.037291,0.880724
4,Global Single Model (54 Backbone + 80 CTX-head),0.759103,0.049998,0.049020,0.009840,0.037341,0.880411
5,"Clustering_V0_Full_k2 (Winner c0=0, c1=10 + 80...",0.751557,0.050775,0.049719,0.010301,0.037942,0.877271


## Section 4: Accelerated SHAP Feature Importance Analysis

Compute SHAP values for all 6 models using XGBoost's native `pred_contribs=True` tree traversal. For MoE models, SHAP is computed per-cluster and aligned into unified feature matrices.

Features are categorized as **Tabular**, **CTX (160-dim)**, or **Head Hidden (80-dim)** to quantify attribution shares.


In [5]:
from eval_hybrid.shap_analysis import run_full_shap_analysis

shap_info = run_full_shap_analysis(eval_global, eval_v0, results_map, artifacts_dir)

print()
print("===== TOP 15 SHAP FEATURES COMPARISON =====")
print(shap_info["df_top20"].head(15).to_string())


Executing Accelerated SHAP Feature Importance Analysis


[SHAP] Computing SHAP values for: Global Single Model (54 Backbone)...


  -> Done in 1.468s (pred_contribs)


[SHAP] Computing SHAP values for: Clustering_V0_Full_k2 (Winner c0=0, c1=10)...


  -> Done in 1.011s (pred_contribs)


...

[SHAP] Computing SHAP values for: Clustering_V0_Full_k2 (Winner c0=0, c1=10 + 80 CTX-head)...


  -> Done in 1.055s (pred_contribs)


[SHAP] Saved summary table to /scratch/user/u.rp352032/MDR-Project/notebooks/experiment/derived_8.4-hybrid-lstm-1.2/artifacts/shap_importance_summary.csv


[SHAP] Saved visualization plot to /scratch/user/u.rp352032/MDR-Project/notebooks/experiment/derived_8.4-hybrid-lstm-1.2/artifacts/shap_summary_plots.png



===== TOP 15 SHAP FEATURES COMPARISON =====
   Global Single Model (54 Backbone)_feature  Global Single Model (54 Backbone)_shap_val Clustering_V0_Full_k2 (Winner c0=0, c1=10)_feature  Clustering_V0_Full_k2 (Winner c0=0, c1=10)_shap_val Global Single Model (54 Backbone + 160 CTX)_feature  Global Single Model (54 Backbone + 160 CTX)_shap_val Clustering_V0_Full_k2 (Winner c0=0, c1=10 + 160 CTX)_feature  Clustering_V0_Full_k2 (Winner c0=0, c1=10 + 160 CTX)_shap_val Global Single Model (54 Backbone + 80 CTX-head)_feature  Global Single Model (54 Backbone + 80 CTX-head)_shap_val Clustering_V0_Full_k2 (Winner c0=0, c1=10 + 80 CTX-head)_feature  Clustering_V0_Full_k2 (Winner c0=0, c1=10 + 80 CTX-head)_shap_val
0                 V_rollmin_LST_modis_kobs30                                    0.030712                             V_ema_LST_modis_kobs30                                             0.029229                                              ctx_24                                          

## Section 5: LSTM Feature Importance & Acceleration Summary

Quantify SHAP attribution for **Tabular** vs **CTX (160-dim)** vs **Head Hidden (80-dim)** features across all 6 models. Compare whether the 80-dim bottleneck concentrates importance more effectively than the full 160-dim CTX.

Also display the SHAP computation speedup from C++/CUDA `pred_contribs=True` over `TreeExplainer`.


In [6]:
ctx_tab_rows = []
for model_name, info in shap_info["ctx_vs_tabular"].items():
    ctx_tab_rows.append({
        "Model Name": model_name,
        "Tabular Features": info["num_tabular_features"],
        "LSTM Features": info["num_ctx_features"],
        "Tabular SHAP Sum": round(info["tabular_shap_sum"], 4),
        "LSTM SHAP Sum": round(info["ctx_shap_sum"], 4),
        "Tabular % Share": f"{info['tabular_pct']:.2f}%",
        "LSTM % Share": f"{info['ctx_pct']:.2f}%",
    })

print()
print("===== TABULAR vs LSTM FEATURE IMPORTANCE SHARE =====")
print(pd.DataFrame(ctx_tab_rows).to_string())

print()
print("===== SHAP COMPUTATION TIME (pred_contribs) =====")
for model_name, res in shap_info["shap_results"].items():
    print(f"  {model_name}: {res['shap_calc_time_s']:.3f}s")



===== TABULAR vs LSTM FEATURE IMPORTANCE SHARE =====
                                                 Model Name  Tabular Features  LSTM Features  Tabular SHAP Sum  LSTM SHAP Sum Tabular % Share LSTM % Share
0                         Global Single Model (54 Backbone)                54              0            0.1726         0.0000         100.00%        0.00%
1                Clustering_V0_Full_k2 (Winner c0=0, c1=10)                64              0            0.1713         0.0000         100.00%        0.00%
2               Global Single Model (54 Backbone + 160 CTX)                54            160            0.0382         0.1202          24.09%       75.91%
3      Clustering_V0_Full_k2 (Winner c0=0, c1=10 + 160 CTX)                64            160            0.0410         0.1188          25.67%       74.33%
4           Global Single Model (54 Backbone + 80 CTX-head)                54             80            0.0522         0.1019          33.88%       66.12%
5  Clustering_V0